In [1]:
import os
os.environ['HF_HOME']='/workspace/AAIPL/hf_models/'
from unsloth import FastLanguageModel
import torch

# ═══ Configuration ═══
max_seq_length = 2048   # A-Agent: max_new_tokens=512, but input can be long
dtype = torch.bfloat16  # BF16 for ROCm / MI300X
load_in_4bit = False    # Full precision — we have 192GB VRAM

# Change to local path if models are pre-downloaded
# model_name = "/path/to/models--unsloth--gpt-oss-20b-BF16/snapshots/..."
model_name = "unsloth/gpt-oss-20b-BF16"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=True,
)

print(f"Model loaded: {model_name}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
#### Unsloth: `hf_xet==1.1.10` and `ipykernel>6.30.1` breaks progress bars. Disabling for now in XET.
#### Unsloth: To re-enable progress bars, please downgrade to `ipykernel==6.30.1` or wait for a fix to
https://github.com/huggingface/xet-core/issues/526
INFO 02-14 15:51:20 [__init__.py:225] Automatically detected platform rocm.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.10.9: Fast Gpt_Oss patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Tri

Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Model loaded: unsloth/gpt-oss-20b-BF16
Parameter count: 20,914,757,184


In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                        # LoRA rank (higher = more capacity)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,               # 2x rank for faster convergence
    lora_dropout=0,              # 0 is optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Making `model.base_model.model.model` require gradients
Trainable: 15,925,248 / 20,930,682,432 (0.08%)


In [3]:
import json
from datasets import Dataset

# ═══ Load dataset ═══
DATA_DIR = "/workspace/AAIPL/hf_models/hub/datasets--Aayushktyagi--SFT_Apti/snapshots/f5199fdcda9db63487aeca763201aa4c05ef11f2"  # Adjust path as needed on remote server

with open(f"{DATA_DIR}/aagent_chatml_train.json") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data):,} samples")
print(f"Val:   {len(val_data):,} samples")
print(f"Sample keys: {list(train_data[0].keys())}")
print(f"\nSample messages[0]:")
for msg in train_data[0]['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

Train: 114,351 samples
Val:   12,705 samples
Sample keys: ['messages']

Sample messages[0]:
  [system]: You are a logical reasoning expert. Answer the given multiple-choice question.
Provide your answer a...
  [user]: 5 persons D, F, U, P, Q are seated in a row (left to right). D and P sit next to each other. There a...
  [assistant]: {"answer": "B", "reasoning": "Arrangement: D - P - Q - U - F. Answer: D."}...


In [4]:
# Convert to HuggingFace Datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

Train dataset: Dataset({
    features: ['messages'],
    num_rows: 114351
})
Val dataset:   Dataset({
    features: ['messages'],
    num_rows: 12705
})


In [5]:
batch = val_dataset[0]
batch.keys()

dict_keys(['messages'])

In [6]:
batch['messages']

[{'content': 'You are a logical reasoning expert. Answer the given multiple-choice question.\nProvide your answer as a JSON object with "answer" (letter A/B/C/D) and "reasoning" (your step-by-step reasoning).',
  'role': 'system'},
 {'content': 'Identify the next element in the sequence: HV, JT, LR, NP, PN, RL, ?\n\nA) TH\nB) TK\nC) TI\nD) TJ',
  'role': 'user'},
 {'content': '{"answer": "D", "reasoning": "First letters +2, second letters -2."}',
  'role': 'assistant'}]

In [7]:
from unsloth.chat_templates import standardize_sharegpt

# Standardize message format (handles role name normalization)
train_dataset = standardize_sharegpt(train_dataset)
val_dataset = standardize_sharegpt(val_dataset)

# Formatting function: apply model's native chat template
def formatting_prompts_func(examples):
    # standardize_sharegpt may rename 'messages' to 'conversations'
    key = "conversations" if "conversations" in examples else "messages"
    convos = examples[key]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

# Preview formatted text
print("═" * 60)
print("Sample formatted training text:")
print("═" * 60)
print(train_dataset["text"][0][:500])
print("...")

Map:   0%|          | 0/114351 [00:00<?, ? examples/s]

Map:   0%|          | 0/12705 [00:00<?, ? examples/s]

════════════════════════════════════════════════════════════
Sample formatted training text:
════════════════════════════════════════════════════════════
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-02-14

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a logical reasoning expert. Answer the given multiple-choice question.
Provide your answer as a JSON object with "answer" (le
...


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import DataCollatorForSeq2Seq

# ═══ Training Configuration ═══
OUTPUT_DIR = "aagent_gptoss_outputs"
NUM_EPOCHS = 1        # Start with 1, increase if val loss still dropping
BATCH_SIZE = 128       # MI300X can handle more; tune based on seq_length
GRAD_ACCUM = 8        # Effective batch = 128 * 8 = 1024
LEARNING_RATE = 2e-4

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    packing=False,  # Don't pack — preserve conversation boundaries
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
        dataloader_num_workers=0,  # For ROCm stability
        remove_unused_columns=True,
    ),
)

# Train ONLY on assistant responses — mask system + user tokens from loss
# NOTE: gpt-oss uses <|channel|>final between "assistant" and "<|message|>"
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>final<|message|>",
)

print(f"Training config:")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total train samples: {len(train_dataset):,}")
print(f"  Total val samples:   {len(val_dataset):,}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Steps/epoch: ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/114351 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/12705 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/114351 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/12705 [00:00<?, ? examples/s]

Training config:
  Effective batch size: 1024
  Total train samples: 114,351
  Total val samples:   12,705
  Epochs: 1
  Steps/epoch: ~111


In [10]:
gpu_stats = torch.cuda.get_device_properties(0)
reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_mem = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM: {total_mem} GB")
print(f"Reserved before training: {reserved} GB")
print(f"Available for KV cache + training: {total_mem - reserved:.1f} GB")

GPU: 
Total VRAM: 255.69 GB
Reserved before training: 39.46 GB
Available for KV cache + training: 216.2 GB


In [11]:
# Switch to training mode
FastLanguageModel.for_training(model)

# Train
trainer_stats = trainer.train()

# Print final stats
print(f"\nTraining completed!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"  Final loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
peak_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"  Peak VRAM: {peak_mem} GB / {total_mem} GB ({100*peak_mem/total_mem:.1f}%)")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998}.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,0.494800,0.132906
50,0.057400,0.044862
75,0.033000,0.030959
100,0.034400,0.027996


Unsloth: Not an error, but GptOssForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



Training completed!
  Runtime: 6801s (113.4 min)
  Final loss: 0.4934461109805852
  Peak VRAM: 251.72 GB / 255.69 GB (98.4%)


In [15]:
# Save LoRA adapters (small, ~100-300MB)
# LORA_PATH = "aagent_gptoss_lora"
LORA_PATH="aagent_gptoss_outputs/checkpoint-112"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"LoRA adapters saved to: {LORA_PATH}")

# Save merged 16-bit model (full size, for vLLM deployment)
MERGED_PATH = "aagent_gptoss_merged_16bit_checkpoint-112"
print(f"Saving merged 16-bit model to: {MERGED_PATH} ...")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to: {MERGED_PATH}")

LoRA adapters saved to: aagent_gptoss_outputs/checkpoint-112
Saving merged 16-bit model to: aagent_gptoss_merged_16bit_checkpoint-112 ...
Found HuggingFace hub cache directory: /workspace/AAIPL/hf_models/hub
Checking cache directory for required files...


Unsloth: Copying 9 files from cache to `aagent_gptoss_merged_16bit_checkpoint-112`: 100%|██████████| 9/9 [00:59<00:00,  6.62s/it]


Successfully copied all 9 files from cache to `aagent_gptoss_merged_16bit_checkpoint-112`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 9/9 [01:44<00:00, 11.58s/it]


Unsloth: Merge process complete. Saved to `/workspace/AAIPL/aagent_gptoss_merged_16bit_checkpoint-112`
Merged model saved to: aagent_gptoss_merged_16bit_checkpoint-112


In [ ]:
# ═══ Load trained model (for fresh session / separate eval) ═══
# Skip this cell if model is already in memory from training above.
# Choose ONE of the two methods below:

# ── Method 1: Load base + LoRA adapters (smaller files, fast load) ──
from unsloth import FastLanguageModel
import torch

LORA_PATH = "aagent_gptoss_outputs/checkpoint-112"  # or "aagent_gptoss_lora"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    trust_remote_code=True,
)
print(f"✓ Loaded LoRA model from: {LORA_PATH}")

# # ── Method 2: Load merged 16-bit model (full weights, no adapter overhead) ──
# MERGED_PATH = "aagent_gptoss_merged_16bit_checkpoint-112"
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MERGED_PATH,
#     max_seq_length=2048,
#     dtype=torch.bfloat16,
#     load_in_4bit=False,
#     trust_remote_code=True,
# )
# print(f"✓ Loaded merged model from: {MERGED_PATH}")

# Switch to fast inference mode
FastLanguageModel.for_inference(model)
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model device: {next(model.parameters()).device}")

In [16]:
# Switch to inference mode (2x faster)
FastLanguageModel.for_inference(model)

# Test questions — one per category
test_questions = [
    # Syllogism
    """Statements:
1. All dogs are animals.
2. All animals are living beings.
Conclusions:
I. All dogs are living beings.
II. Some living beings are dogs.

A) Only conclusion I follows
B) Only conclusion II follows
C) Both conclusions I and II follow
D) Neither conclusion I nor II follows""",

    # Blood Relations
    """A is the father of B. B is the mother of C. D is the brother of A. What is D to C?

A) Grandfather
B) Uncle
C) Great Uncle
D) Father""",

    # Seating Arrangement
    """5 persons A, B, C, D, E are seated in a row (left to right). 
B is to the immediate right of A. 
D is at one of the ends. 
C is to the immediate left of E.
Who is in the middle?

A) A
B) B
C) C
D) E""",

    # Series
    """Find the next term in the series: 2, 6, 18, 54, ?

A) 108
B) 162
C) 180
D) 216""",
]

system_prompt = (
    "You are a logical reasoning expert. Answer the given multiple-choice question.\n"
    'Provide your answer as a JSON object with "answer" (letter A/B/C/D) and "reasoning" (your step-by-step reasoning).'
)

for i, q in enumerate(test_questions):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    print(f"\n{'═'*50}")
    print(f"Q{i+1}: {q[:80]}...")
    print(f"A{i+1}: {response[:300]}")


══════════════════════════════════════════════════
Q1: Statements:
1. All dogs are animals.
2. All animals are living beings.
Conclusio...
A1: answer">B</assistant><json>{"answer": "B", "reasoning": "Conclusion I ('All dogs are living beings') follows. Conclusion II ('Some living beings are dogs') does not follow."}

══════════════════════════════════════════════════
Q2: A is the father of B. B is the mother of C. D is the brother of A. What is D to ...
A2: answer_dict, reasoning_dict{"answer": "B", "reasoning": "Following the family relationship chain in the story, D is the uncle of C. Answer: B."}

══════════════════════════════════════════════════
Q3: 5 persons A, B, C, D, E are seated in a row (left to right). 
B is to the immedi...
A3: answer">B</assistantreasoning">Arrangement: A - B - C - E - D. Answer: B.</answer>

══════════════════════════════════════════════════
Q4: Find the next term in the series: 2, 6, 18, 54, ?

A) 108
B) 162
C) 180
D) 216...
A4: answer">B</assistant><c

In [ ]:
import json
from collections import defaultdict
import numpy as np

# ═══ Full Validation Accuracy (BATCHED) + Confusion Metrics ═══
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_raw = json.load(f)

print(f"Evaluating on FULL validation set: {len(val_raw)} samples\n")

FastLanguageModel.for_inference(model)

EVAL_BATCH_SIZE = 32  # Tune based on VRAM headroom
CLASSES = ["A", "B", "C", "D"]
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# ── Structured output: JSON schema enforcement (like API response_format) ──
# Uses `outlines` JSONLogitsProcessor to constrain generation to valid JSON
# matching our schema — equivalent to response_format={"type":"json_object"}
from pydantic import BaseModel, Field
from typing import Literal

class AAgentResponse(BaseModel):
    """Expected A-Agent output schema."""
    answer: Literal["A", "B", "C", "D"]
    reasoning: str

json_processor = None
try:
    from outlines.processors import JSONLogitsProcessor
    json_processor = JSONLogitsProcessor(
        schema=AAgentResponse,
        tokenizer=tokenizer,
        whitespace_pattern=r"[\n\t ]*",  # Allow flexible whitespace
    )
    print("✓ Using outlines JSONLogitsProcessor (structured JSON output enforced)")
except ImportError:
    print("⚠ outlines not installed — pip install outlines")
    print("  Falling back to unconstrained generation + post-hoc JSON parsing")
except Exception as e:
    print(f"⚠ outlines init failed: {e}")
    print("  Falling back to unconstrained generation + post-hoc JSON parsing")

# ── Generation config ──
GEN_CONFIG = dict(
    max_new_tokens=512,    # A-Agent: competition limit is 512 tokens / ≤9s
    temperature=0.1,       # Low temp for near-deterministic eval
    do_sample=True,        # Enable sampling (required for top_p)
    top_p=0.9,             # Nucleus sampling — trims low-prob tail
    repetition_penalty=1.2,# Penalise repeated tokens / degenerate loops
)

# ── Step 1: Pre-process all samples ──
samples = []
skipped = 0

for item in val_raw:
    msgs = item["messages"]
    gt_response = msgs[-1]["content"]
    try:
        gt_parsed = json.loads(gt_response)
        gt_answer = gt_parsed["answer"]
    except Exception:
        skipped += 1
        continue

    user_content = msgs[1]["content"] if len(msgs) > 1 else ""
    if "syllogism" in user_content.lower() or "Statements:" in user_content:
        topic_key = "Syllogisms"
    elif "blood" in user_content.lower() or "father" in user_content.lower() or "mother" in user_content.lower():
        topic_key = "Blood Relations"
    elif "seated" in user_content.lower() or "seating" in user_content.lower():
        topic_key = "Seating Arrangements"
    elif "series" in user_content.lower() or "next term" in user_content.lower() or "Find the" in user_content:
        topic_key = "Series"
    else:
        topic_key = "Other"

    samples.append({
        "gt_answer": gt_answer,
        "topic_key": topic_key,
        "prompt_msgs": msgs[:-1],
    })

print(f"Valid samples: {len(samples)} (skipped {skipped} with unparseable GT)")
print(f"Generation config: {GEN_CONFIG}")
print(f"JSON enforcement: {'outlines JSONLogitsProcessor' if json_processor else 'post-hoc parsing'}")

# ── Step 2: Left-pad tokenizer for batched generation ──
orig_padding_side = tokenizer.padding_side
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Step 3: Batched inference ──
correct = 0
total = 0
parse_fails = 0
topic_stats = defaultdict(lambda: {"correct": 0, "total": 0})

# Confusion matrix: rows = ground truth, cols = predicted
confusion = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
# Per-topic confusion matrices
topic_confusion = defaultdict(lambda: np.zeros((len(CLASSES), len(CLASSES)), dtype=int))

num_batches = (len(samples) + EVAL_BATCH_SIZE - 1) // EVAL_BATCH_SIZE

for batch_idx in range(num_batches):
    start = batch_idx * EVAL_BATCH_SIZE
    end = min(start + EVAL_BATCH_SIZE, len(samples))
    batch = samples[start:end]

    prompt_texts = [
        tokenizer.apply_chat_template(
            s["prompt_msgs"], add_generation_prompt=True, tokenize=False
        )
        for s in batch
    ]
    inputs = tokenizer(
        prompt_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_seq_length,
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    # Build generate kwargs — attach logits_processor if available
    gen_kwargs = dict(
        **inputs,
        **GEN_CONFIG,
        pad_token_id=tokenizer.pad_token_id,
    )
    if json_processor is not None:
        gen_kwargs["logits_processor"] = [json_processor]

    outputs = model.generate(**gen_kwargs)

    for i, s in enumerate(batch):
        response = tokenizer.decode(
            outputs[i][prompt_lengths:], skip_special_tokens=True
        )

        pred = None
        # Primary: parse full JSON → extract "answer"
        try:
            parsed = json.loads(response)
            pred = parsed["answer"]
        except Exception:
            # Fallback 1: regex-style key extraction from partial JSON
            for letter in CLASSES:
                if f'"answer": "{letter}"' in response or f'"answer":"{letter}"' in response:
                    pred = letter
                    break
            # Fallback 2: first A-D character in response
            if pred is None:
                for ch in response.strip()[:20]:
                    if ch in "ABCD":
                        pred = ch
                        break

        if pred and pred in class_to_idx and s["gt_answer"] in class_to_idx:
            total += 1
            gt_idx = class_to_idx[s["gt_answer"]]
            pred_idx = class_to_idx[pred]
            confusion[gt_idx][pred_idx] += 1
            topic_confusion[s["topic_key"]][gt_idx][pred_idx] += 1
            topic_stats[s["topic_key"]]["total"] += 1
            if pred == s["gt_answer"]:
                correct += 1
                topic_stats[s["topic_key"]]["correct"] += 1
        else:
            parse_fails += 1

    processed = end
    if processed % 500 < EVAL_BATCH_SIZE or batch_idx == num_batches - 1:
        acc = 100 * correct / total if total > 0 else 0
        print(f"  [{processed}/{len(samples)}] accuracy: {correct}/{total} = {acc:.1f}%  parse_fails: {parse_fails}")

tokenizer.padding_side = orig_padding_side

# ═══════════════════════════════════════════════════════════════
# RESULTS
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"FULL VALIDATION RESULTS ({len(val_raw)} samples)")
print(f"{'═'*60}")
print(f"Overall Accuracy: {correct}/{total} = {100*correct/total:.2f}%")
print(f"Parse failures: {parse_fails}  |  GT skipped: {skipped}")

# ── Per-Topic Breakdown ──
print(f"\n{'─'*60}")
print(f"Per-Topic Accuracy:")
print(f"{'─'*60}")
for topic, stats in sorted(topic_stats.items()):
    t_acc = 100 * stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"  {topic:25s}: {stats['correct']:5d}/{stats['total']:5d} = {t_acc:.1f}%")

# ── Overall Confusion Matrix ──
print(f"\n{'─'*60}")
print(f"Confusion Matrix (rows=GT, cols=Pred):")
print(f"{'─'*60}")
header = "       " + "".join(f"  Pred {c}" for c in CLASSES)
print(header)
for r, cls in enumerate(CLASSES):
    row_str = f"  GT {cls}: " + "".join(f"{confusion[r][c]:7d}" for c in range(len(CLASSES)))
    print(row_str)

# ── Per-Class Precision / Recall / F1 ──
print(f"\n{'─'*60}")
print(f"Per-Class Metrics:")
print(f"{'─'*60}")
print(f"  {'Class':>5s}  {'Precision':>9s}  {'Recall':>9s}  {'F1':>9s}  {'Support':>7s}")
macro_p, macro_r, macro_f1 = 0, 0, 0
n_classes_present = 0
for idx, cls in enumerate(CLASSES):
    tp = confusion[idx][idx]
    fn = confusion[idx].sum() - tp       # GT=cls but predicted other
    fp = confusion[:, idx].sum() - tp    # predicted=cls but GT is other
    support = confusion[idx].sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"  {cls:>5s}  {precision:>9.4f}  {recall:>9.4f}  {f1:>9.4f}  {support:>7d}")
    if support > 0:
        macro_p += precision
        macro_r += recall
        macro_f1 += f1
        n_classes_present += 1

if n_classes_present > 0:
    print(f"  {'Macro':>5s}  {macro_p/n_classes_present:>9.4f}  {macro_r/n_classes_present:>9.4f}  {macro_f1/n_classes_present:>9.4f}")

# ── Per-Topic Confusion Matrices + Class Metrics ──
print(f"\n{'═'*60}")
print(f"PER-TOPIC CONFUSION MATRICES & CLASS METRICS")
print(f"{'═'*60}")

for topic in sorted(topic_confusion.keys()):
    cm = topic_confusion[topic]
    t_total = cm.sum()
    t_correct = sum(cm[i][i] for i in range(len(CLASSES)))
    t_acc = 100 * t_correct / t_total if t_total > 0 else 0

    print(f"\n┌─ {topic} (n={t_total}, acc={t_acc:.1f}%) ─┐")
    print("       " + "".join(f"  Pred {c}" for c in CLASSES))
    for r, cls in enumerate(CLASSES):
        row_str = f"  GT {cls}: " + "".join(f"{cm[r][c]:7d}" for c in range(len(CLASSES)))
        print(row_str)

    # Class metrics for this topic
    print(f"  {'Class':>5s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'N':>5s}")
    for idx, cls in enumerate(CLASSES):
        tp = cm[idx][idx]
        fn = cm[idx].sum() - tp
        fp = cm[:, idx].sum() - tp
        support = cm[idx].sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        if support > 0:
            print(f"  {cls:>5s}  {prec:>6.3f}  {rec:>6.3f}  {f1:>6.3f}  {support:>5d}")

  Progress: 50/200 evaluated, accuracy: 70.0%


In [ ]:

import json
from datasets import Dataset

# ═══ Load dataset ═══
DATA_DIR = "/workspace/AAIPL/hf_models/hub/datasets--Aayushktyagi--SFT_Apti/snapshots/f5199fdcda9db63487aeca763201aa4c05ef11f2"  # Adjust path as needed on remote server

with open(f"{DATA_DIR}/aagent_chatml_train.json") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data):,} samples")
print(f"Val:   {len(val_data):,} samples")
print(f"Sample keys: {list(train_data[0].keys())}")
print(f"\nSample messages[0]:")
for msg in train_data[0]['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")


    
import os
# os.environ['HF_HOME']='/workspace/AAIPL/hf_models/'
from unsloth import FastLanguageModel
import torch

# ═══ Configuration ═══
max_seq_length = 2048   # A-Agent: max_new_tokens=512, but input can be long
dtype = torch.bfloat16  # BF16 for ROCm / MI300X
load_in_4bit = False    # Full precision — we have 192GB VRAM

# Change to local path if models are pre-downloaded
# model_name = "/path/to/models--unsloth--gpt-oss-20b-BF16/snapshots/..."
model_name = "/workspace/AAIPL/aagent_gptoss_merged_16bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=True,
)

print(f"Model loaded: {model_name}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")

import json
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm

# ═══ Full Validation Accuracy (BATCHED) + Confusion Metrics ═══
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_raw = json.load(f)

print(f"Evaluating on FULL validation set: {len(val_raw)} samples\n")

FastLanguageModel.for_inference(model)

EVAL_BATCH_SIZE = 32  # Tune based on VRAM headroom
CLASSES = ["A", "B", "C", "D"]
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# ── Structured output: JSON schema enforcement (like API response_format) ──
# Uses `outlines` JSONLogitsProcessor to constrain generation to valid JSON
# matching our schema — equivalent to response_format={"type":"json_object"}
from pydantic import BaseModel, Field
from typing import Literal

class AAgentResponse(BaseModel):
    """Expected A-Agent output schema."""
    answer: Literal["A", "B", "C", "D"]
    reasoning: str

json_processor = None
try:
    from outlines.processors import JSONLogitsProcessor
    json_processor = JSONLogitsProcessor(
        schema=AAgentResponse,
        tokenizer=tokenizer,
        whitespace_pattern=r"[\n\t ]*",  # Allow flexible whitespace
    )
    print("✓ Using outlines JSONLogitsProcessor (structured JSON output enforced)")
except ImportError:
    print("⚠ outlines not installed — pip install outlines")
    print("  Falling back to unconstrained generation + post-hoc JSON parsing")
except Exception as e:
    print(f"⚠ outlines init failed: {e}")
    print("  Falling back to unconstrained generation + post-hoc JSON parsing")

# ── Generation config ──
GEN_CONFIG = dict(
    max_new_tokens=512,    # A-Agent: competition limit is 512 tokens / ≤9s
    temperature=0.1,       # Low temp for near-deterministic eval
    do_sample=True,        # Enable sampling (required for top_p)
    top_p=0.9,             # Nucleus sampling — trims low-prob tail
    repetition_penalty=1.2,# Penalise repeated tokens / degenerate loops
)

# ── Step 1: Pre-process all samples ──
samples = []
skipped = 0

for item in val_raw:
    msgs = item["messages"]
    gt_response = msgs[-1]["content"]
    try:
        gt_parsed = json.loads(gt_response)
        gt_answer = gt_parsed["answer"]
    except Exception:
        skipped += 1
        continue

    user_content = msgs[1]["content"] if len(msgs) > 1 else ""
    if "syllogism" in user_content.lower() or "Statements:" in user_content:
        topic_key = "Syllogisms"
    elif "blood" in user_content.lower() or "father" in user_content.lower() or "mother" in user_content.lower():
        topic_key = "Blood Relations"
    elif "seated" in user_content.lower() or "seating" in user_content.lower():
        topic_key = "Seating Arrangements"
    elif "series" in user_content.lower() or "next term" in user_content.lower() or "Find the" in user_content:
        topic_key = "Series"
    else:
        topic_key = "Other"

    samples.append({
        "gt_answer": gt_answer,
        "topic_key": topic_key,
        "prompt_msgs": msgs[:-1],
    })

print(f"Valid samples: {len(samples)} (skipped {skipped} with unparseable GT)")
print(f"Generation config: {GEN_CONFIG}")
print(f"JSON enforcement: {'outlines JSONLogitsProcessor' if json_processor else 'post-hoc parsing'}")

# ── Step 2: Left-pad tokenizer for batched generation ──
orig_padding_side = tokenizer.padding_side
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Step 3: Batched inference ──
correct = 0
total = 0
parse_fails = 0
topic_stats = defaultdict(lambda: {"correct": 0, "total": 0})

# Confusion matrix: rows = ground truth, cols = predicted
confusion = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
# Per-topic confusion matrices
topic_confusion = defaultdict(lambda: np.zeros((len(CLASSES), len(CLASSES)), dtype=int))

num_batches = (len(samples) + EVAL_BATCH_SIZE - 1) // EVAL_BATCH_SIZE

for batch_idx in tqdm(range(num_batches)):
    start = batch_idx * EVAL_BATCH_SIZE
    end = min(start + EVAL_BATCH_SIZE, len(samples))
    batch = samples[start:end]

    prompt_texts = [
        tokenizer.apply_chat_template(
            s["prompt_msgs"], add_generation_prompt=True, tokenize=False
        )
        for s in batch
    ]
    inputs = tokenizer(
        prompt_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_seq_length,
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    # Build generate kwargs — attach logits_processor if available
    gen_kwargs = dict(
        **inputs,
        **GEN_CONFIG,
        pad_token_id=tokenizer.pad_token_id,
    )
    if json_processor is not None:
        gen_kwargs["logits_processor"] = [json_processor]

    outputs = model.generate(**gen_kwargs)

    for i, s in enumerate(batch):
        response = tokenizer.decode(
            outputs[i][prompt_lengths:], skip_special_tokens=True
        )

        pred = None
        # Primary: parse full JSON → extract "answer"
        try:
            parsed = json.loads(response)
            pred = parsed["answer"]
        except Exception:
            # Fallback 1: regex-style key extraction from partial JSON
            for letter in CLASSES:
                if f'"answer": "{letter}"' in response or f'"answer":"{letter}"' in response:
                    pred = letter
                    break
            # Fallback 2: first A-D character in response
            if pred is None:
                for ch in response.strip()[:20]:
                    if ch in "ABCD":
                        pred = ch
                        break

        if pred and pred in class_to_idx and s["gt_answer"] in class_to_idx:
            total += 1
            gt_idx = class_to_idx[s["gt_answer"]]
            pred_idx = class_to_idx[pred]
            confusion[gt_idx][pred_idx] += 1
            topic_confusion[s["topic_key"]][gt_idx][pred_idx] += 1
            topic_stats[s["topic_key"]]["total"] += 1
            if pred == s["gt_answer"]:
                correct += 1
                topic_stats[s["topic_key"]]["correct"] += 1
        else:
            parse_fails += 1

    processed = end
    if processed % 500 < EVAL_BATCH_SIZE or batch_idx == num_batches - 1:
        acc = 100 * correct / total if total > 0 else 0
        print(f"  [{processed}/{len(samples)}] accuracy: {correct}/{total} = {acc:.1f}%  parse_fails: {parse_fails}")

tokenizer.padding_side = orig_padding_side

# ═══════════════════════════════════════════════════════════════
# RESULTS
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"FULL VALIDATION RESULTS ({len(val_raw)} samples)")
print(f"{'═'*60}")
print(f"Overall Accuracy: {correct}/{total} = {100*correct/total:.2f}%")
print(f"Parse failures: {parse_fails}  |  GT skipped: {skipped}")

# ── Per-Topic Breakdown ──
print(f"\n{'─'*60}")
print(f"Per-Topic Accuracy:")
print(f"{'─'*60}")
for topic, stats in sorted(topic_stats.items()):
    t_acc = 100 * stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"  {topic:25s}: {stats['correct']:5d}/{stats['total']:5d} = {t_acc:.1f}%")

# ── Overall Confusion Matrix ──
print(f"\n{'─'*60}")
print(f"Confusion Matrix (rows=GT, cols=Pred):")
print(f"{'─'*60}")
header = "       " + "".join(f"  Pred {c}" for c in CLASSES)
print(header)
for r, cls in enumerate(CLASSES):
    row_str = f"  GT {cls}: " + "".join(f"{confusion[r][c]:7d}" for c in range(len(CLASSES)))
    print(row_str)

# ── Per-Class Precision / Recall / F1 ──
print(f"\n{'─'*60}")
print(f"Per-Class Metrics:")
print(f"{'─'*60}")
print(f"  {'Class':>5s}  {'Precision':>9s}  {'Recall':>9s}  {'F1':>9s}  {'Support':>7s}")
macro_p, macro_r, macro_f1 = 0, 0, 0
n_classes_present = 0
for idx, cls in enumerate(CLASSES):
    tp = confusion[idx][idx]
    fn = confusion[idx].sum() - tp       # GT=cls but predicted other
    fp = confusion[:, idx].sum() - tp    # predicted=cls but GT is other
    support = confusion[idx].sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"  {cls:>5s}  {precision:>9.4f}  {recall:>9.4f}  {f1:>9.4f}  {support:>7d}")
    if support > 0:
        macro_p += precision
        macro_r += recall
        macro_f1 += f1
        n_classes_present += 1

if n_classes_present > 0:
    print(f"  {'Macro':>5s}  {macro_p/n_classes_present:>9.4f}  {macro_r/n_classes_present:>9.4f}  {macro_f1/n_classes_present:>9.4f}")

# ── Per-Topic Confusion Matrices + Class Metrics ──
print(f"\n{'═'*60}")
print(f"PER-TOPIC CONFUSION MATRICES & CLASS METRICS")
print(f"{'═'*60}")

for topic in sorted(topic_confusion.keys()):
    cm = topic_confusion[topic]
    t_total = cm.sum()
    t_correct = sum(cm[i][i] for i in range(len(CLASSES)))
    t_acc = 100 * t_correct / t_total if t_total > 0 else 0

    print(f"\n┌─ {topic} (n={t_total}, acc={t_acc:.1f}%) ─┐")
    print("       " + "".join(f"  Pred {c}" for c in CLASSES))
    for r, cls in enumerate(CLASSES):
        row_str = f"  GT {cls}: " + "".join(f"{cm[r][c]:7d}" for c in range(len(CLASSES)))
        print(row_str)

    # Class metrics for this topic
    print(f"  {'Class':>5s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'N':>5s}")
    for idx, cls in enumerate(CLASSES):
        tp = cm[idx][idx]
        fn = cm[idx].sum() - tp
        fp = cm[:, idx].sum() - tp
        support = cm[idx].sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        if support > 0:
            print(f"  {cls:>5s}  {prec:>6.3f}  {rec:>6.3f}  {f1:>6.3f}  {support:>5d}")

In [ ]:
import os
os.environ['HF_HOME']='/workspace/AAIPL/hf_models/'
from unsloth import FastLanguageModel
import torch

# ═══ Configuration ═══
max_seq_length = 2048   # A-Agent: max_new_tokens=512, but input can be long
dtype = torch.bfloat16  # BF16 for ROCm / MI300X
load_in_4bit = False    # Full precision — we have 192GB VRAM

# Change to local path if models are pre-downloaded
# model_name = "/path/to/models--unsloth--gpt-oss-20b-BF16/snapshots/..."
model_name = "Qwen/Qwen2.5-14B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=True,
)

# Fix: decoder-only models need left-padding for correct batched generation
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded: {model_name}")
print(f"Padding side: {tokenizer.padding_side}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")


model = FastLanguageModel.get_peft_model(
    model,
    r=32,                        # LoRA rank (higher = more capacity)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,               # 2x rank for faster convergence
    lora_dropout=0,              # 0 is optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


import json
from datasets import Dataset

# ═══ Load dataset ═══
DATA_DIR = "/workspace/AAIPL/hf_models/hub/datasets--Aayushktyagi--SFT_Apti/snapshots/46edfc3805a28c5d1e25884cfa3a6605dcf24a10"  # Adjust path as needed on remote server

with open(f"{DATA_DIR}/aagent_chatml_train.json") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data):,} samples")
print(f"Val:   {len(val_data):,} samples")
print(f"Sample keys: {list(train_data[0].keys())}")
print(f"\nSample messages[0]:")
for msg in train_data[0]['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")



# Convert to HuggingFace Datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")



from unsloth.chat_templates import standardize_sharegpt

# Standardize message format (handles role name normalization)
train_dataset = standardize_sharegpt(train_dataset)
val_dataset = standardize_sharegpt(val_dataset)

# Formatting function: apply model's native chat template
def formatting_prompts_func(examples):
    # standardize_sharegpt may rename 'messages' to 'conversations'
    key = "conversations" if "conversations" in examples else "messages"
    convos = examples[key]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

# Preview formatted text
print("═" * 60)
print("Sample formatted training text:")
print("═" * 60)
print(train_dataset["text"][0][:500])
print("...")




from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import DataCollatorForSeq2Seq

# ═══ Training Configuration ═══
OUTPUT_DIR = "aagent_qwen25_14_Inst_outputs"
NUM_EPOCHS = 1        # Start with 1, increase if val loss still dropping
BATCH_SIZE = 128       # MI300X can handle more; tune based on seq_length
GRAD_ACCUM = 8        # Effective batch = 8 * 8 = 64
LEARNING_RATE = 2e-4

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    packing=False,  # Don't pack — preserve conversation boundaries
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
        dataloader_num_workers=0,  # For ROCm stability
        remove_unused_columns=True,
    ),
)



# # Train ONLY on assistant responses — mask system + user tokens from loss
# trainer = train_on_responses_only(
#     trainer,
#     instruction_part="<|start|>user<|message|>",
#     # response_part="<|start|>assistant<|message|>",
#     response_part="<|start|>assistant<|channel|>final<|message|>",
# )

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print(f"Training config:")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total train samples: {len(train_dataset):,}")
print(f"  Total val samples:   {len(val_dataset):,}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Steps/epoch: ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")



gpu_stats = torch.cuda.get_device_properties(0)
reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_mem = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM: {total_mem} GB")
print(f"Reserved before training: {reserved} GB")
print(f"Available for KV cache + training: {total_mem - reserved:.1f} GB")


# Save LoRA adapters (small, ~100-300MB)
LORA_PATH = "aagent_qwen25_14_Inst_lora"
# LORA_PATH="aagent_gptoss_outputs/checkpoint-112"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"LoRA adapters saved to: {LORA_PATH}")

# Save merged 16-bit model (full size, for vLLM deployment)
MERGED_PATH = "aagent_qwen25_14_Inst_merged_16bit"
print(f"Saving merged 16-bit model to: {MERGED_PATH} ...")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to: {MERGED_PATH}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Qwen2.5-14B-Instruct A-Agent — Full Batched Evaluation
# ═══════════════════════════════════════════════════════════════
# Assumes cell 16 has already loaded & trained the Qwen model.
# If starting a fresh session, uncomment the model-loading block below.

# ── (Optional) Load trained Qwen model in a fresh session ──
# import os
# os.environ['HF_HOME'] = '/workspace/AAIPL/hf_models/'
# from unsloth import FastLanguageModel
# import torch
# MODEL_PATH = "aagent_qwen25_14_Inst_outputs"  # or "aagent_qwen25_14_Inst_merged_16bit"
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MODEL_PATH, max_seq_length=2048,
#     dtype=torch.bfloat16, load_in_4bit=False, trust_remote_code=True,
# )
# print(f"✓ Loaded Qwen A-Agent from: {MODEL_PATH}")

import json
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm

# ═══ Load validation data ═══
DATA_DIR = "/workspace/AAIPL/hf_models/hub/datasets--Aayushktyagi--SFT_Apti/snapshots/46edfc3805a28c5d1e25884cfa3a6605dcf24a10"

with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_raw = json.load(f)

print(f"Evaluating Qwen2.5-14B A-Agent on validation set: {len(val_raw)} samples\n")

FastLanguageModel.for_inference(model)

EVAL_BATCH_SIZE = 32
CLASSES = ["A", "B", "C", "D"]
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# ── Structured output via outlines (optional) ──
from pydantic import BaseModel
from typing import Literal

class AAgentResponse(BaseModel):
    answer: Literal["A", "B", "C", "D"]
    reasoning: str

json_processor = None
try:
    from outlines.processors import JSONLogitsProcessor
    json_processor = JSONLogitsProcessor(
        schema=AAgentResponse,
        tokenizer=tokenizer,
        whitespace_pattern=r"[\n\t ]*",
    )
    print("✓ Using outlines JSONLogitsProcessor (structured JSON output enforced)")
except ImportError:
    print("⚠ outlines not installed — falling back to post-hoc JSON parsing")
except Exception as e:
    print(f"⚠ outlines init failed: {e} — falling back to post-hoc JSON parsing")

# ── Generation config ──
GEN_CONFIG = dict(
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.2,
)

# ── Pre-process samples ──
samples = []
skipped = 0

for item in val_raw:
    msgs = item["messages"]
    gt_response = msgs[-1]["content"]
    try:
        gt_parsed = json.loads(gt_response)
        gt_answer = gt_parsed["answer"]
    except Exception:
        skipped += 1
        continue

    user_content = msgs[1]["content"] if len(msgs) > 1 else ""
    # Topic classification from content keywords
    ul = user_content.lower()
    if "syllogism" in ul or "statement" in ul and "conclusion" in ul:
        topic_key = "Syllogisms"
    elif any(kw in ul for kw in ["blood", "father", "mother", "brother", "sister", "son", "daughter", "uncle", "family"]):
        topic_key = "Blood Relations"
    elif any(kw in ul for kw in ["seated", "seating", "sitting", "row", "circle", "table", "arrangement"]):
        topic_key = "Seating Arrangements"
    elif any(kw in ul for kw in ["series", "next term", "pattern", "sequence", "find the"]):
        topic_key = "Series & Patterns"
    else:
        topic_key = "Other"

    samples.append({
        "gt_answer": gt_answer,
        "topic_key": topic_key,
        "prompt_msgs": msgs[:-1],
    })

print(f"Valid samples: {len(samples)} (skipped {skipped} with unparseable GT)")
print(f"Generation config: {GEN_CONFIG}")
print(f"JSON enforcement: {'outlines' if json_processor else 'post-hoc parsing'}")

# ── Left-pad for batched generation ──
orig_padding_side = tokenizer.padding_side
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Batched inference ──
correct = 0
total = 0
parse_fails = 0
topic_stats = defaultdict(lambda: {"correct": 0, "total": 0})
confusion = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
topic_confusion = defaultdict(lambda: np.zeros((len(CLASSES), len(CLASSES)), dtype=int))

num_batches = (len(samples) + EVAL_BATCH_SIZE - 1) // EVAL_BATCH_SIZE

for batch_idx in tqdm(range(num_batches), desc="Qwen Eval"):
    start = batch_idx * EVAL_BATCH_SIZE
    end = min(start + EVAL_BATCH_SIZE, len(samples))
    batch = samples[start:end]

    prompt_texts = [
        tokenizer.apply_chat_template(
            s["prompt_msgs"], add_generation_prompt=True, tokenize=False
        )
        for s in batch
    ]
    inputs = tokenizer(
        prompt_texts, return_tensors="pt", padding=True,
        truncation=True, max_length=2048,
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    gen_kwargs = dict(**inputs, **GEN_CONFIG, pad_token_id=tokenizer.pad_token_id)
    if json_processor is not None:
        gen_kwargs["logits_processor"] = [json_processor]

    outputs = model.generate(**gen_kwargs)

    for i, s in enumerate(batch):
        response = tokenizer.decode(
            outputs[i][prompt_lengths:], skip_special_tokens=True
        )

        pred = None
        # Primary: full JSON parse
        try:
            parsed = json.loads(response)
            pred = parsed.get("answer", "")
            if pred and len(pred) >= 1:
                pred = pred[0].upper()
        except Exception:
            pass
        # Fallback 1: regex key extraction
        if pred not in CLASSES:
            import re
            m = re.search(r'"answer"\s*:\s*"([A-D])"', response)
            if m:
                pred = m.group(1)
        # Fallback 2: first A-D in response
        if pred not in CLASSES:
            for ch in response.strip()[:30]:
                if ch in "ABCD":
                    pred = ch
                    break

        if pred in CLASSES and s["gt_answer"] in class_to_idx:
            total += 1
            gt_idx = class_to_idx[s["gt_answer"]]
            pred_idx = class_to_idx[pred]
            confusion[gt_idx][pred_idx] += 1
            topic_confusion[s["topic_key"]][gt_idx][pred_idx] += 1
            topic_stats[s["topic_key"]]["total"] += 1
            if pred == s["gt_answer"]:
                correct += 1
                topic_stats[s["topic_key"]]["correct"] += 1
        else:
            parse_fails += 1

    processed = end
    if processed % 500 < EVAL_BATCH_SIZE or batch_idx == num_batches - 1:
        acc = 100 * correct / total if total > 0 else 0
        print(f"  [{processed}/{len(samples)}] acc: {correct}/{total} = {acc:.1f}%  parse_fails: {parse_fails}")

tokenizer.padding_side = orig_padding_side

# ═══════════════════════════════════════════════════════════════
# RESULTS
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"QWEN2.5-14B A-AGENT — VALIDATION RESULTS ({len(val_raw)} samples)")
print(f"{'═'*60}")
print(f"Overall Accuracy: {correct}/{total} = {100*correct/total:.2f}%" if total > 0 else "No valid predictions")
print(f"Parse failures: {parse_fails}  |  GT skipped: {skipped}")

# ── Per-Topic Breakdown ──
print(f"\n{'─'*60}")
print(f"Per-Topic Accuracy:")
print(f"{'─'*60}")
for topic, stats in sorted(topic_stats.items()):
    t_acc = 100 * stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"  {topic:25s}: {stats['correct']:5d}/{stats['total']:5d} = {t_acc:.1f}%")

# ── Confusion Matrix ──
print(f"\n{'─'*60}")
print(f"Confusion Matrix (rows=GT, cols=Pred):")
print(f"{'─'*60}")
header = "       " + "".join(f"  Pred {c}" for c in CLASSES)
print(header)
for r, cls in enumerate(CLASSES):
    row_str = f"  GT {cls}: " + "".join(f"{confusion[r][c]:7d}" for c in range(len(CLASSES)))
    print(row_str)

# ── Per-Class Precision / Recall / F1 ──
print(f"\n{'─'*60}")
print(f"Per-Class Metrics:")
print(f"{'─'*60}")
print(f"  {'Class':>5s}  {'Precision':>9s}  {'Recall':>9s}  {'F1':>9s}  {'Support':>7s}")
macro_p, macro_r, macro_f1 = 0, 0, 0
n_classes_present = 0
for idx, cls in enumerate(CLASSES):
    tp = confusion[idx][idx]
    fn = confusion[idx].sum() - tp
    fp = confusion[:, idx].sum() - tp
    support = confusion[idx].sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"  {cls:>5s}  {precision:>9.4f}  {recall:>9.4f}  {f1:>9.4f}  {support:>7d}")
    if support > 0:
        macro_p += precision
        macro_r += recall
        macro_f1 += f1
        n_classes_present += 1

if n_classes_present > 0:
    print(f"  {'Macro':>5s}  {macro_p/n_classes_present:>9.4f}  {macro_r/n_classes_present:>9.4f}  {macro_f1/n_classes_present:>9.4f}")

# ── Per-Topic Confusion Matrices ──
print(f"\n{'═'*60}")
print(f"PER-TOPIC CONFUSION MATRICES & CLASS METRICS")
print(f"{'═'*60}")

for topic in sorted(topic_confusion.keys()):
    cm = topic_confusion[topic]
    t_total = cm.sum()
    t_correct = sum(cm[i][i] for i in range(len(CLASSES)))
    t_acc = 100 * t_correct / t_total if t_total > 0 else 0

    print(f"\n┌─ {topic} (n={int(t_total)}, acc={t_acc:.1f}%) ─┐")
    print("       " + "".join(f"  Pred {c}" for c in CLASSES))
    for r, cls in enumerate(CLASSES):
        row_str = f"  GT {cls}: " + "".join(f"{cm[r][c]:7.0f}" for c in range(len(CLASSES)))
        print(row_str)

    print(f"  {'Class':>5s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'N':>5s}")
    for idx, cls in enumerate(CLASSES):
        tp = cm[idx][idx]
        fn = cm[idx].sum() - tp
        fp = cm[:, idx].sum() - tp
        support = cm[idx].sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        if support > 0:
            print(f"  {cls:>5s}  {prec:>6.3f}  {rec:>6.3f}  {f1:>6.3f}  {int(support):>5d}")

In [ ]:

import json
from datasets import Dataset

# ═══ Load dataset ═══
DATA_DIR = "/workspace/AAIPL/hf_models/hub/datasets--Aayushktyagi--SFT_Apti/snapshots/f5199fdcda9db63487aeca763201aa4c05ef11f2"  # Adjust path as needed on remote server

with open(f"{DATA_DIR}/aagent_chatml_train.json") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data):,} samples")
print(f"Val:   {len(val_data):,} samples")
print(f"Sample keys: {list(train_data[0].keys())}")
print(f"\nSample messages[0]:")
for msg in train_data[0]['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")


    
import os
# os.environ['HF_HOME']='/workspace/AAIPL/hf_models/'
from unsloth import FastLanguageModel
import torch

# ═══ Configuration ═══
max_seq_length = 2048   # A-Agent: max_new_tokens=512, but input can be long
dtype = torch.bfloat16  # BF16 for ROCm / MI300X
load_in_4bit = False    # Full precision — we have 192GB VRAM

# Change to local path if models are pre-downloaded
# model_name = "/path/to/models--unsloth--gpt-oss-20b-BF16/snapshots/..."
model_name = "/workspace/AAIPL/aagent_gptoss_merged_16bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=True,
)

print(f"Model loaded: {model_name}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")

import json
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm

# ═══ Full Validation Accuracy (BATCHED) + Confusion Metrics ═══
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_raw = json.load(f)

print(f"Evaluating on FULL validation set: {len(val_raw)} samples\n")

FastLanguageModel.for_inference(model)

EVAL_BATCH_SIZE = 32  # Tune based on VRAM headroom
CLASSES = ["A", "B", "C", "D"]
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# ── Structured output: JSON schema enforcement (like API response_format) ──
# Uses `outlines` JSONLogitsProcessor to constrain generation to valid JSON
# matching our schema — equivalent to response_format={"type":"json_object"}
from pydantic import BaseModel, Field
from typing import Literal

class AAgentResponse(BaseModel):
    """Expected A-Agent output schema."""
    answer: Literal["A", "B", "C", "D"]
    reasoning: str

json_processor = None
try:
    from outlines.processors import JSONLogitsProcessor
    json_processor = JSONLogitsProcessor(
        schema=AAgentResponse,
        tokenizer=tokenizer,
        whitespace_pattern=r"[\n\t ]*",  # Allow flexible whitespace
    )
    print("✓ Using outlines JSONLogitsProcessor (structured JSON output enforced)")
except ImportError:
    print("⚠ outlines not installed — pip install outlines")
    print("  Falling back to unconstrained generation + post-hoc JSON parsing")
except Exception as e:
    print(f"⚠ outlines init failed: {e}")
    print("  Falling back to unconstrained generation + post-hoc JSON parsing")

# ── Generation config ──
GEN_CONFIG = dict(
    max_new_tokens=512,    # A-Agent: competition limit is 512 tokens / ≤9s
    temperature=0.1,       # Low temp for near-deterministic eval
    do_sample=True,        # Enable sampling (required for top_p)
    top_p=0.9,             # Nucleus sampling — trims low-prob tail
    repetition_penalty=1.2,# Penalise repeated tokens / degenerate loops
)

# ── Step 1: Pre-process all samples ──
samples = []
skipped = 0

for item in val_raw:
    msgs = item["messages"]
    gt_response = msgs[-1]["content"]
    try:
        gt_parsed = json.loads(gt_response)
        gt_answer = gt_parsed["answer"]
    except Exception:
        skipped += 1
        continue

    user_content = msgs[1]["content"] if len(msgs) > 1 else ""
    if "syllogism" in user_content.lower() or "Statements:" in user_content:
        topic_key = "Syllogisms"
    elif "blood" in user_content.lower() or "father" in user_content.lower() or "mother" in user_content.lower():
        topic_key = "Blood Relations"
    elif "seated" in user_content.lower() or "seating" in user_content.lower():
        topic_key = "Seating Arrangements"
    elif "series" in user_content.lower() or "next term" in user_content.lower() or "Find the" in user_content:
        topic_key = "Series"
    else:
        topic_key = "Other"

    samples.append({
        "gt_answer": gt_answer,
        "topic_key": topic_key,
        "prompt_msgs": msgs[:-1],
    })

print(f"Valid samples: {len(samples)} (skipped {skipped} with unparseable GT)")
print(f"Generation config: {GEN_CONFIG}")
print(f"JSON enforcement: {'outlines JSONLogitsProcessor' if json_processor else 'post-hoc parsing'}")

# ── Step 2: Left-pad tokenizer for batched generation ──
orig_padding_side = tokenizer.padding_side
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Step 3: Batched inference ──
correct = 0
total = 0
parse_fails = 0
topic_stats = defaultdict(lambda: {"correct": 0, "total": 0})

# Confusion matrix: rows = ground truth, cols = predicted
confusion = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
# Per-topic confusion matrices
topic_confusion = defaultdict(lambda: np.zeros((len(CLASSES), len(CLASSES)), dtype=int))

num_batches = (len(samples) + EVAL_BATCH_SIZE - 1) // EVAL_BATCH_SIZE

for batch_idx in tqdm(range(num_batches)):
    start = batch_idx * EVAL_BATCH_SIZE
    end = min(start + EVAL_BATCH_SIZE, len(samples))
    batch = samples[start:end]

    prompt_texts = [
        tokenizer.apply_chat_template(
            s["prompt_msgs"], add_generation_prompt=True, tokenize=False
        )
        for s in batch
    ]
    inputs = tokenizer(
        prompt_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_seq_length,
    ).to(model.device)

    prompt_lengths = inputs["input_ids"].shape[1]

    # Build generate kwargs — attach logits_processor if available
    gen_kwargs = dict(
        **inputs,
        **GEN_CONFIG,
        pad_token_id=tokenizer.pad_token_id,
    )
    if json_processor is not None:
        gen_kwargs["logits_processor"] = [json_processor]

    outputs = model.generate(**gen_kwargs)

    for i, s in enumerate(batch):
        response = tokenizer.decode(
            outputs[i][prompt_lengths:], skip_special_tokens=True
        )

        pred = None
        # Primary: parse full JSON → extract "answer"
        try:
            parsed = json.loads(response)
            pred = parsed["answer"]
        except Exception:
            # Fallback 1: regex-style key extraction from partial JSON
            for letter in CLASSES:
                if f'"answer": "{letter}"' in response or f'"answer":"{letter}"' in response:
                    pred = letter
                    break
            # Fallback 2: first A-D character in response
            if pred is None:
                for ch in response.strip()[:20]:
                    if ch in "ABCD":
                        pred = ch
                        break

        if pred and pred in class_to_idx and s["gt_answer"] in class_to_idx:
            total += 1
            gt_idx = class_to_idx[s["gt_answer"]]
            pred_idx = class_to_idx[pred]
            confusion[gt_idx][pred_idx] += 1
            topic_confusion[s["topic_key"]][gt_idx][pred_idx] += 1
            topic_stats[s["topic_key"]]["total"] += 1
            if pred == s["gt_answer"]:
                correct += 1
                topic_stats[s["topic_key"]]["correct"] += 1
        else:
            parse_fails += 1

    processed = end
    if processed % 500 < EVAL_BATCH_SIZE or batch_idx == num_batches - 1:
        acc = 100 * correct / total if total > 0 else 0
        print(f"  [{processed}/{len(samples)}] accuracy: {correct}/{total} = {acc:.1f}%  parse_fails: {parse_fails}")

tokenizer.padding_side = orig_padding_side

# ═══════════════════════════════════════════════════════════════
# RESULTS
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"FULL VALIDATION RESULTS ({len(val_raw)} samples)")
print(f"{'═'*60}")
print(f"Overall Accuracy: {correct}/{total} = {100*correct/total:.2f}%")
print(f"Parse failures: {parse_fails}  |  GT skipped: {skipped}")

# ── Per-Topic Breakdown ──
print(f"\n{'─'*60}")
print(f"Per-Topic Accuracy:")
print(f"{'─'*60}")
for topic, stats in sorted(topic_stats.items()):
    t_acc = 100 * stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"  {topic:25s}: {stats['correct']:5d}/{stats['total']:5d} = {t_acc:.1f}%")

# ── Overall Confusion Matrix ──
print(f"\n{'─'*60}")
print(f"Confusion Matrix (rows=GT, cols=Pred):")
print(f"{'─'*60}")
header = "       " + "".join(f"  Pred {c}" for c in CLASSES)
print(header)
for r, cls in enumerate(CLASSES):
    row_str = f"  GT {cls}: " + "".join(f"{confusion[r][c]:7d}" for c in range(len(CLASSES)))
    print(row_str)

# ── Per-Class Precision / Recall / F1 ──
print(f"\n{'─'*60}")
print(f"Per-Class Metrics:")
print(f"{'─'*60}")
print(f"  {'Class':>5s}  {'Precision':>9s}  {'Recall':>9s}  {'F1':>9s}  {'Support':>7s}")
macro_p, macro_r, macro_f1 = 0, 0, 0
n_classes_present = 0
for idx, cls in enumerate(CLASSES):
    tp = confusion[idx][idx]
    fn = confusion[idx].sum() - tp       # GT=cls but predicted other
    fp = confusion[:, idx].sum() - tp    # predicted=cls but GT is other
    support = confusion[idx].sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"  {cls:>5s}  {precision:>9.4f}  {recall:>9.4f}  {f1:>9.4f}  {support:>7d}")
    if support > 0:
        macro_p += precision
        macro_r += recall
        macro_f1 += f1
        n_classes_present += 1

if n_classes_present > 0:
    print(f"  {'Macro':>5s}  {macro_p/n_classes_present:>9.4f}  {macro_r/n_classes_present:>9.4f}  {macro_f1/n_classes_present:>9.4f}")

# ── Per-Topic Confusion Matrices + Class Metrics ──
print(f"\n{'═'*60}")
print(f"PER-TOPIC CONFUSION MATRICES & CLASS METRICS")
print(f"{'═'*60}")

for topic in sorted(topic_confusion.keys()):
    cm = topic_confusion[topic]
    t_total = cm.sum()
    t_correct = sum(cm[i][i] for i in range(len(CLASSES)))
    t_acc = 100 * t_correct / t_total if t_total > 0 else 0

    print(f"\n┌─ {topic} (n={t_total}, acc={t_acc:.1f}%) ─┐")
    print("       " + "".join(f"  Pred {c}" for c in CLASSES))
    for r, cls in enumerate(CLASSES):
        row_str = f"  GT {cls}: " + "".join(f"{cm[r][c]:7d}" for c in range(len(CLASSES)))
        print(row_str)

    # Class metrics for this topic
    print(f"  {'Class':>5s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'N':>5s}")
    for idx, cls in enumerate(CLASSES):
        tp = cm[idx][idx]
        fn = cm[idx].sum() - tp
        fp = cm[:, idx].sum() - tp
        support = cm[idx].sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        if support > 0:
            print(f"  {cls:>5s}  {prec:>6.3f}  {rec:>6.3f}  {f1:>6.3f}  {support:>5d}")